# Comparación de heurísticas — Scientometrics

Ejecute desde la carpeta donde descomprimió este paquete. Utiliza los mismos 112 pares del test temporal de la Tabla 2.
Las métricas incluidas ya se calcularon con el script adjunto; este notebook permite reproducirlas.
No se vuelve a entrenar ningún clasificador. Dependencias: pandas, numpy y scikit-learn.

Se conserva el estado exacto de los grafos experimentales, incluidos 79 autoenlaces en la unión histórica. El grado cuenta cada autoenlace dos veces, como en NetworkX. Eliminarlos requeriría regenerar las características y reevaluar también los clasificadores.


In [ ]:
from pathlib import Path
import hashlib
import json
import math
import re
import sys
import zipfile

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
import sklearn


def read_snapshot(text, structural=False):
    """Read the flat node/edge records in the supplied anonymized GML snapshots."""
    nodes = {}
    authors = set()
    for block in re.findall(r'^  node \[\n(.*?)^  \]', text, re.M | re.S):
        node_id = int(re.search(r'^    id (\d+)\s*$', block, re.M).group(1))
        label = re.search(r'^    label "([^"\n]*)"\s*$', block, re.M).group(1)
        assert node_id not in nodes
        nodes[node_id] = label
        if re.search(r'^    tipo "autor"\s*$', block, re.M):
            authors.add(label)
    assert len(nodes) == len(set(nodes.values()))
    if not structural:
        return authors
    assert not re.search(r'^  (directed|multigraph) 1\s*$', text, re.M)
    adjacency = {name: set() for name in nodes.values()}
    edge_count = 0
    for block in re.findall(r'^  edge \[\n(.*?)^  \]', text, re.M | re.S):
        a = int(re.search(r'^    source (\d+)\s*$', block, re.M).group(1))
        b = int(re.search(r'^    target (\d+)\s*$', block, re.M).group(1))
        u, v = nodes[a], nodes[b]
        assert v not in adjacency[u], 'Parallel edge requires separate treatment.'
        adjacency[u].add(v)
        adjacency[v].add(u)
        edge_count += 1
    assert edge_count == len(re.findall(r'^  edge \[', text, re.M))
    assert len(nodes) == len(re.findall(r'^  node \[', text, re.M))
    return adjacency


def degree(graph, node):
    # Match undirected NetworkX degree: a self-loop contributes twice.
    return len(graph[node]) + int(node in graph[node])


def edge_count(graph):
    return sum(degree(graph, node) for node in graph) // 2


def canonical_pairs(frame):
    return [tuple(sorted((str(a), str(b)))) for a, b in frame[['source', 'target']].itertuples(index=False, name=None)]


def main(input_dir, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    input_names = {
        'graphs': 'gml_anonimos_reproducibles.zip',
        'train': 'sample_temporal_train_hist2000_2020_target2021_2022_thr1_neg1_random_isolates(1).csv',
        'val': 'sample_temporal_val_hist2000_2020_target2021_2022_thr1_neg1_random_isolates(1).csv',
        'test': 'sample_temporal_test_hist2000_2022_target2023_2024_thr1_neg1_random_isolates(1).csv',
        'predictions': 'P5-1_Scores_Balanceados_FINAL_test_predictions(4).csv',
    }
    data = {k: pd.read_csv(input_dir / v) for k, v in input_names.items() if k != 'graphs'}
    dev = pd.concat([data['train'], data['val']], ignore_index=True)
    test = data['test']
    preds = data['predictions']
    assert len(dev) == 34 and dev['connected'].sum() == 17
    assert len(test) == 112 and test['connected'].sum() == 56
    assert len(set(canonical_pairs(test))) == len(test)
    assert not set(canonical_pairs(dev)) & set(canonical_pairs(test))
    assert dict(zip(canonical_pairs(test), test['connected'])) == dict(zip(canonical_pairs(preds), preds['connected']))

    with zipfile.ZipFile(input_dir / input_names['graphs']) as z:
        def read_by_name(name):
            matches = [n for n in z.namelist() if Path(n).name == name]
            assert len(matches) == 1, (name, matches)
            return z.read(matches[0]).decode('utf-8')
        g0 = read_snapshot(read_by_name('BD_UNAM_Sub_Graph_2000_2020.gml'), True)
        g1 = read_snapshot(read_by_name('BD_UNAM_Sub_Graph_2021_2022.gml'), True)
        known0 = read_snapshot(read_by_name('BD_UNAM_Graph_2000_2020.gml'))
        known1 = read_snapshot(read_by_name('BD_UNAM_Graph_2021_2022.gml'))
    assert (len(g0), edge_count(g0)) == (526, 933)
    assert (len(g1), edge_count(g1)) == (1064, 2250)
    hist = {u: set(neighbors) for u, neighbors in g0.items()}
    for u, neighbors in g1.items():
        hist.setdefault(u, set()).update(neighbors)
    known = known0 | known1
    assert set(test['source']) | set(test['target']) <= known
    for author in known:
        hist.setdefault(author, set())

    score_rows = []
    for u, v, y in test[['source', 'target', 'connected']].itertuples(index=False, name=None):
        assert v not in hist[u], 'Candidate already linked in historical structural graph.'
        common = (hist[u] & hist[v]) - {u, v}
        score_rows.append({
            'source': u, 'target': v, 'connected': int(y),
            'Common Neighbors': len(common),
            'Adamic--Adar': sum(1.0 / math.log(degree(hist, z)) for z in common),
            'Resource Allocation': sum(1.0 / degree(hist, z) for z in common),
            'Preferential Attachment': degree(hist, u) * degree(hist, v),
        })
    scores = pd.DataFrame(score_rows)
    correspondence = {'Adamic--Adar': 'adamic_adar_index', 'Resource Allocation': 'resource_allocation', 'Preferential Attachment': 'preferential_attachment'}
    checks = {}
    for method, original in correspondence.items():
        error = float(np.max(np.abs(scores[method].to_numpy() - test[original].to_numpy())))
        checks[original] = error
        np.testing.assert_allclose(scores[method], test[original], rtol=1e-12, atol=1e-12)

    rows = []
    for method in score_rows[0].keys() - {'source', 'target', 'connected'}:
        values = scores[method]
        rows.append({'method': method, 'ROC_AUC': roc_auc_score(scores['connected'], values), 'AP': average_precision_score(scores['connected'], values), 'n_test': len(test), 'n_positive': int(test['connected'].sum()), 'n_zero_scores': int((values == 0).sum())})
    metrics = pd.DataFrame(rows).sort_values('AP', ascending=False).reset_index(drop=True)
    model_rows = []
    for col in preds:
        if col.endswith('__score'):
            model_rows.append({'method': col.removesuffix('__score'), 'ROC_AUC': roc_auc_score(preds['connected'], preds[col]), 'AP': average_precision_score(preds['connected'], preds[col])})
    model_metrics = pd.DataFrame(model_rows)
    scores.to_csv(output_dir / 'heuristic_scores_test_2023_2024.csv', index=False)
    metrics.to_csv(output_dir / 'heuristic_metrics_test_2023_2024.csv', index=False)
    model_metrics.to_csv(output_dir / 'classifier_ranking_metrics_verified.csv', index=False)
    audit = {'test_horizon': '2023-2024', 'feature_cutoff': 2022, 'negative_ratio': '1:1', 'n_test': len(test), 'n_positive': int(test['connected'].sum()), 'n_negative': int((test['connected'] == 0).sum()), 'structural_nodes_upto_2022': len(hist), 'structural_edges_upto_2022': edge_count(hist), 'self_loops_preserved': sum(u in hist[u] for u in hist), 'baseline_hyperparameter_search': False, 'threshold_selection': False, 'feature_recomputation_max_absolute_errors': checks, 'versions': {'python': sys.version.split()[0], 'numpy': np.__version__, 'pandas': pd.__version__, 'scikit_learn': sklearn.__version__}, 'input_sha256': {name: hashlib.sha256((input_dir / name).read_bytes()).hexdigest() for name in input_names.values()}}
    (output_dir / 'baseline_audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
    print(metrics.to_string(index=False))
    print(model_metrics.to_string(index=False))
    print(json.dumps(audit, indent=2))




In [ ]:
main(Path('baseline_inputs/Linkpred'), Path('baseline_results'))
